In [11]:
import cv2
import numpy as np
import pyttsx3
import threading
import time

from tensorflow import keras
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from cvzone.HandTrackingModule import HandDetector
from collections import Counter


# ==========================================
# HAND SIGN DETECTOR
# ==========================================

class HandSignDetector:


    def __init__(
        self,
        model_path="sign_alphabet_mob_model.keras",
        class_indices_path="class_indices_mob.npy"
    ):

        print("Loading MobileNetV2 model...")

        self.model = keras.models.load_model(
            model_path,
            compile=False
        )

        print("Model loaded successfully!")


        class_indices = np.load(
            class_indices_path,
            allow_pickle=True
        ).item()


        self.idx_to_class = {
            v:k for k,v in class_indices.items()
        }


        print("\nClass Mapping:")
        print(self.idx_to_class)



        self.detector = HandDetector(
            maxHands=2,
            detectionCon=0.7
        )


        self.img_size = 224


        # prediction smoothing
        self.prediction_buffer = []


        # voice
        self.engine = pyttsx3.init()

        self.last_spoken = ""
        self.last_speak_time = 0



    # ==========================================
    # TEXT TO SPEECH
    # ==========================================

    def speak(self, text):

      now = time.time()

    # avoid repeating too quickly
      if now - self.last_speak_time < 1.5:
        return

      self.last_speak_time = now
      self.last_spoken = text

      def voice_thread(letter):
    
            try:
    
                engine = pyttsx3.init()
    
                engine.say(letter)
    
                engine.runAndWait()
    
                engine.stop()
    
            except Exception as e:
    
                print("Voice error:", e)



            threading.Thread(
            target=voice_thread,
            args=(text,),
            daemon=True
    ).start()


    # ==========================================
    # PREPROCESS IMAGE
    # ==========================================

    def preprocess_image(self, img_crop):


        canvas = np.zeros(
            (self.img_size,self.img_size,3),
            dtype=np.uint8
        )


        h,w = img_crop.shape[:2]


        if h == 0 or w == 0:
            return None,None



        scale = min(
            self.img_size/w,
            self.img_size/h
        )


        new_w = int(w*scale)
        new_h = int(h*scale)



        resized = cv2.resize(
            img_crop,
            (new_w,new_h)
        )


        x_offset = (self.img_size-new_w)//2
        y_offset = (self.img_size-new_h)//2



        canvas[
            y_offset:y_offset+new_h,
            x_offset:x_offset+new_w
        ] = resized



        img_preprocessed = preprocess_input(
            canvas.astype(np.float32)
        )


        img_input = np.expand_dims(
            img_preprocessed,
            axis=0
        )


        return img_input,canvas




    # ==========================================
    # PREDICT
    # ==========================================

    def predict(self,img):


        try:

           result = self.detector.findHands(
           img,
           draw=False
           )


        # cvzone versions return different formats
           if isinstance(result, tuple):

              hands = result[0]
              img = result[1]

           else:

              hands = result



           if hands is None or len(hands) == 0:
             return None, 0, None, None



           all_x=[]
           all_y=[]


           for hand in hands:

                for lm in hand["lmList"]:

                    all_x.append(int(lm[0]))
                    all_y.append(int(lm[1]))



           pad=40
    
    
           frame_h,frame_w = img.shape[:2]
    
    
           x1=max(
                0,
                min(all_x)-pad
            )
    
           y1=max(
                0,
                min(all_y)-pad
            )


           x2=min(
                frame_w,
                max(all_x)+pad
            )
    
    
           y2=min(
                frame_h,
                max(all_y)+pad
            )
    
    
    
           img_crop = img[
                y1:y2,
                x1:x2
            ]


           if img_crop.size==0:
    
                return None,0,None,None
    
    
    
           img_input,processed_img = self.preprocess_image(
                img_crop
            )
    
    
           prediction = self.model.predict(
                img_input,
                verbose=0
                )
    


           index=np.argmax(
                prediction[0]
            )


           confidence=float(
                prediction[0][index]
            )


           label=self.idx_to_class[index]



            # ==============================
            # DEBUG TOP 3
            # ==============================

           top3=np.argsort(
                prediction[0]
            )[-3:][::-1]


           print("\nTop 3 Predictions:")

           for i in top3:

                print(
                    f"{self.idx_to_class[i]} : {prediction[0][i]:.4f}"
                )



            # ==============================
            # SMOOTHING
            # ==============================


           self.prediction_buffer.append(
                label
            )


           if len(self.prediction_buffer)>5:

                self.prediction_buffer.pop(0)



           stable_label = Counter(
                self.prediction_buffer
            ).most_common(1)[0][0]



           return (
                stable_label,
                confidence,
                (x1,y1,x2,y2),
                processed_img
            )



        except Exception as e:

            print(
                "Prediction error:",
                e
            )

            return None,0,None,None





# ==========================================
# MAIN
# ==========================================


def main():


    cap=cv2.VideoCapture(0)


    cap.set(
        cv2.CAP_PROP_FRAME_WIDTH,
        1280
    )

    cap.set(
        cv2.CAP_PROP_FRAME_HEIGHT,
        720
    )



    detector=HandSignDetector()



    while True:


        success,img=cap.read()


        if not success:
            continue



        img=cv2.flip(
            img,
            1
        )



        label,confidence,bbox,processed_img = detector.predict(
            img
        )



        if bbox is not None:


            x1,y1,x2,y2=bbox


            cv2.rectangle(
                img,
                (x1,y1),
                (x2,y2),
                (0,255,0),
                2
            )



        if label is not None:


            # overlay alphabet

            cv2.putText(
                img,
                 f"Alphabet: {label}",
                 (img.shape[1]-350, 70),
                 cv2.FONT_HERSHEY_SIMPLEX,
                 1.5,
                (0,255,0),
                 3
            )


            cv2.putText(
                img,
                 f"Confidence: {confidence*100:.1f}%",
                 (img.shape[1]-350, 130),
                 cv2.FONT_HERSHEY_SIMPLEX,
                 0.8,
                (0,255,255),
                 2
            )



            # speak stable alphabet

            detector.speak(label)




        cv2.imshow(
            "Sign Alphabet Recognition",
            img
        )



        if processed_img is not None:

            cv2.imshow(
                "Model Input",
                processed_img
            )



        key=cv2.waitKey(1)


        if key==ord('q'):

            break



    cap.release()

    cv2.destroyAllWindows()



if __name__=="__main__":

    main()

Loading MobileNetV2 model...
Model loaded successfully!

Class Mapping:
{0: 'a', 1: 'b', 2: 'c', 3: 'd', 4: 'e', 5: 'f', 6: 'g', 7: 'h', 8: 'i', 9: 'j', 10: 'k', 11: 'l', 12: 'm', 13: 'n', 14: 'o', 15: 'p', 16: 'q', 17: 'r', 18: 's', 19: 't', 20: 'u', 21: 'v', 22: 'w', 23: 'x', 24: 'y', 25: 'z'}

Top 3 Predictions:
m : 0.5515
j : 0.2708
b : 0.1654

Top 3 Predictions:
a : 0.6365
g : 0.2696
h : 0.0799

Top 3 Predictions:
a : 0.7141
m : 0.2564
g : 0.0100

Top 3 Predictions:
a : 0.9972
w : 0.0028
b : 0.0000

Top 3 Predictions:
a : 1.0000
o : 0.0000
w : 0.0000

Top 3 Predictions:
a : 0.9701
g : 0.0282
h : 0.0007

Top 3 Predictions:
a : 0.9974
g : 0.0023
o : 0.0001

Top 3 Predictions:
a : 0.9994
g : 0.0004
w : 0.0001

Top 3 Predictions:
a : 1.0000
w : 0.0000
o : 0.0000

Top 3 Predictions:
a : 0.9994
u : 0.0002
x : 0.0002

Top 3 Predictions:
a : 0.9997
w : 0.0003
o : 0.0000

Top 3 Predictions:
a : 0.9995
h : 0.0002
x : 0.0002

Top 3 Predictions:
a : 0.9976
g : 0.0020
h : 0.0002

Top 3 Predict

In [ ]:
def speak(self, text):

    now = time.time()

    # don't repeat same letter too quickly
    if now - self.last_speak_time < 1.5:
        return


    self.last_speak_time = now
    self.last_spoken = text


    def voice_thread():

        try:

            engine = pyttsx3.init()

            engine.say(text)

            engine.runAndWait()

            engine.stop()

        except Exception as e:

            print("Voice error:", e)


    threading.Thread(
        target=voice_thread,
        daemon=True
    ).start()

In [ ]:
def speak(self, text):

    now = time.time()

    # avoid repeating too quickly
    if now - self.last_speak_time < 1.5:
        return

    self.last_speak_time = now


    def voice_thread(letter):

        try:

            engine = pyttsx3.init()

            engine.say(letter)

            engine.runAndWait()

            engine.stop()

        except Exception as e:

            print("Voice error:", e)



    threading.Thread(
        target=voice_thread,
        args=(text,),
        daemon=True
    ).start()

In [14]:
import cv2
import numpy as np
import pyttsx3
import threading
import time
import queue

from tensorflow import keras
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from cvzone.HandTrackingModule import HandDetector
from collections import Counter


# ==========================================
# HAND SIGN DETECTOR
# ==========================================

class HandSignDetector:

    def __init__(
        self,
        model_path="sign_alphabet_mob_model.keras",
        class_indices_path="class_indices_mob.npy"
    ):

        print("Loading MobileNetV2 model...")

        self.model = keras.models.load_model(
            model_path,
            compile=False
        )

        print("Model loaded successfully!")


        class_indices = np.load(
            class_indices_path,
            allow_pickle=True
        ).item()


        self.idx_to_class = {
            v:k for k,v in class_indices.items()
        }


        print("\nClass Mapping:")
        print(self.idx_to_class)



        self.detector = HandDetector(
            maxHands=2,
            detectionCon=0.7
        )


        self.img_size = 224


        # prediction smoothing
        self.prediction_buffer = []


        # ===============================
        # VOICE SETUP
        # ===============================

        self.voice_queue = queue.Queue()

        self.last_spoken = ""
        self.last_speak_time = 0


        self.voice_thread = threading.Thread(
            target=self.voice_worker,
            daemon=True
        )

        self.voice_thread.start()



    # ==========================================
    # VOICE WORKER
    # ==========================================

    def voice_worker(self):

        engine = pyttsx3.init()

        while True:

            letter = self.voice_queue.get()


            if letter is None:
                break


            try:

                engine.say(letter)
                engine.runAndWait()


            except Exception as e:

                print("Voice error:",e)


            self.voice_queue.task_done()



    # ==========================================
    # SPEAK
    # ==========================================

    def speak(self, text):

        now = time.time()


        # avoid repeating
        if now - self.last_speak_time < 2:
            return


        if text == self.last_spoken:
            return


        self.last_spoken = text
        self.last_speak_time = now


        self.voice_queue.put(text)



    # ==========================================
    # PREPROCESS
    # ==========================================

    def preprocess_image(self,img_crop):


        canvas = np.zeros(
            (224,224,3),
            dtype=np.uint8
        )


        h,w = img_crop.shape[:2]


        if h==0 or w==0:
            return None,None



        scale = min(
            224/w,
            224/h
        )


        new_w = int(w*scale)
        new_h = int(h*scale)



        resized = cv2.resize(
            img_crop,
            (new_w,new_h)
        )



        x_offset=(224-new_w)//2
        y_offset=(224-new_h)//2



        canvas[
            y_offset:y_offset+new_h,
            x_offset:x_offset+new_w
        ] = resized



        processed = preprocess_input(
            canvas.astype(np.float32)
        )


        processed = np.expand_dims(
            processed,
            axis=0
        )


        return processed,canvas




    # ==========================================
    # PREDICT
    # ==========================================

    def predict(self,img):

        try:


            result = self.detector.findHands(
                img,
                draw=False
            )


            if isinstance(result,tuple):

                hands=result[0]
                img=result[1]

            else:

                hands=result



            if hands is None or len(hands)==0:

                return None,0,None,None



            all_x=[]
            all_y=[]



            for hand in hands:

                for lm in hand["lmList"]:

                    all_x.append(int(lm[0]))
                    all_y.append(int(lm[1]))



            pad=40


            h,w=img.shape[:2]


            x1=max(0,min(all_x)-pad)
            y1=max(0,min(all_y)-pad)

            x2=min(w,max(all_x)+pad)
            y2=min(h,max(all_y)+pad)



            crop = img[
                y1:y2,
                x1:x2
            ]


            if crop.size==0:

                return None,0,None,None



            inp,processed = self.preprocess_image(
                crop
            )


            prediction = self.model.predict(
                inp,
                verbose=0
            )



            index=np.argmax(
                prediction[0]
            )


            confidence=float(
                prediction[0][index]
            )


            label=self.idx_to_class[index]



            # DEBUG

            top3=np.argsort(
                prediction[0]
            )[-3:][::-1]


            print("\nTop 3 Predictions:")

            for i in top3:

                print(
                    f"{self.idx_to_class[i]} : {prediction[0][i]:.4f}"
                )



            # smoothing

            self.prediction_buffer.append(label)


            if len(self.prediction_buffer)>8:

                self.prediction_buffer.pop(0)



            stable_label = Counter(
                self.prediction_buffer
            ).most_common(1)[0][0]



            return (
                stable_label,
                confidence,
                (x1,y1,x2,y2),
                processed
            )



        except Exception as e:

            print("Prediction error:",e)

            return None,0,None,None





# ==========================================
# MAIN
# ==========================================

def main():


    cap=cv2.VideoCapture(0)


    cap.set(
        cv2.CAP_PROP_FRAME_WIDTH,
        1280
    )

    cap.set(
        cv2.CAP_PROP_FRAME_HEIGHT,
        720
    )


    detector=HandSignDetector()



    while True:


        success,img=cap.read()


        if not success:
            continue



        img=cv2.flip(
            img,
            1
        )



        label,confidence,bbox,processed = detector.predict(
            img
        )



        if bbox:


            x1,y1,x2,y2=bbox


            cv2.rectangle(
                img,
                (x1,y1),
                (x2,y2),
                (0,255,0),
                2
            )



        if label:


            # overlay right top

            cv2.putText(
                img,
                f"Alphabet: {label}",
                (img.shape[1]-350,70),
                cv2.FONT_HERSHEY_SIMPLEX,
                1.5,
                (0,255,0),
                3
            )


            cv2.putText(
                img,
                f"Confidence: {confidence*100:.1f}%",
                (img.shape[1]-350,130),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0,255,255),
                2
            )


            # voice

            detector.speak(label)




        cv2.imshow(
            "Sign Alphabet Recognition",
            img
        )



        if processed is not None:

            cv2.imshow(
                "Model Input",
                processed
            )



        if cv2.waitKey(1)==ord('q'):

            break



    cap.release()
    cv2.destroyAllWindows()



if __name__=="__main__":

    main()

Loading MobileNetV2 model...
Model loaded successfully!

Class Mapping:
{0: 'a', 1: 'b', 2: 'c', 3: 'd', 4: 'e', 5: 'f', 6: 'g', 7: 'h', 8: 'i', 9: 'j', 10: 'k', 11: 'l', 12: 'm', 13: 'n', 14: 'o', 15: 'p', 16: 'q', 17: 'r', 18: 's', 19: 't', 20: 'u', 21: 'v', 22: 'w', 23: 'x', 24: 'y', 25: 'z'}

Top 3 Predictions:
b : 0.8391
e : 0.1535
q : 0.0050

Top 3 Predictions:
w : 0.8927
b : 0.1060
x : 0.0007

Top 3 Predictions:
w : 0.9244
x : 0.0413
b : 0.0290

Top 3 Predictions:
w : 0.9866
b : 0.0083
a : 0.0043

Top 3 Predictions:
b : 0.5806
w : 0.2864
x : 0.0889

Top 3 Predictions:
w : 0.4927
b : 0.4784
a : 0.0232

Top 3 Predictions:
w : 0.6761
a : 0.1638
b : 0.1565

Top 3 Predictions:
b : 0.6955
w : 0.2894
p : 0.0093

Top 3 Predictions:
w : 0.9474
b : 0.0434
a : 0.0089

Top 3 Predictions:
w : 0.6250
b : 0.2303
a : 0.1445

Top 3 Predictions:
w : 0.7232
a : 0.1825
x : 0.0664

Top 3 Predictions:
w : 0.9910
b : 0.0090
x : 0.0000

Top 3 Predictions:
w : 0.6822
b : 0.2618
a : 0.0520

Top 3 Predict

In [14]:
import cv2
import numpy as np
import pyttsx3
import threading
import time
import queue

from tensorflow import keras
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from cvzone.HandTrackingModule import HandDetector
from collections import Counter


# ==========================================
# HAND SIGN DETECTOR
# ==========================================

class HandSignDetector:

    def __init__(
        self,
        model_path="sign_alphabet_mob_model.keras",
        class_indices_path="class_indices_mob.npy"
    ):

        print("Loading MobileNetV2 model...")

        self.model = keras.models.load_model(
            model_path,
            compile=False
        )

        print("Model loaded successfully!")


        class_indices = np.load(
            class_indices_path,
            allow_pickle=True
        ).item()


        self.idx_to_class = {
            v:k for k,v in class_indices.items()
        }


        print("\nClass Mapping:")
        print(self.idx_to_class)



        self.detector = HandDetector(
            maxHands=2,
            detectionCon=0.7
        )


        self.img_size = 224


        # prediction smoothing
        self.prediction_buffer = []


        # ===============================
        # VOICE SETUP
        # ===============================

        self.voice_queue = queue.Queue()

        self.last_spoken = ""
        self.last_speak_time = 0


        self.voice_thread = threading.Thread(
            target=self.voice_worker,
            daemon=True
        )

        self.voice_thread.start()



    # ==========================================
    # VOICE WORKER
    # ==========================================

    def voice_worker(self):

        engine = pyttsx3.init()

        while True:

            letter = self.voice_queue.get()


            if letter is None:
                break


            try:

                engine.say(letter)
                engine.runAndWait()


            except Exception as e:

                print("Voice error:",e)


            self.voice_queue.task_done()



    # ==========================================
    # SPEAK
    # ==========================================

    def speak(self, text):

        now = time.time()


        # avoid repeating
        if now - self.last_speak_time < 2:
            return


        if text == self.last_spoken:
            return


        self.last_spoken = text
        self.last_speak_time = now


        self.voice_queue.put(text)



    # ==========================================
    # PREPROCESS
    # ==========================================

    def preprocess_image(self,img_crop):


        canvas = np.zeros(
            (224,224,3),
            dtype=np.uint8
        )


        h,w = img_crop.shape[:2]


        if h==0 or w==0:
            return None,None



        scale = min(
            224/w,
            224/h
        )


        new_w = int(w*scale)
        new_h = int(h*scale)



        resized = cv2.resize(
            img_crop,
            (new_w,new_h)
        )



        x_offset=(224-new_w)//2
        y_offset=(224-new_h)//2



        canvas[
            y_offset:y_offset+new_h,
            x_offset:x_offset+new_w
        ] = resized



        processed = preprocess_input(
            canvas.astype(np.float32)
        )


        processed = np.expand_dims(
            processed,
            axis=0
        )


        return processed,canvas




    # ==========================================
    # PREDICT
    # ==========================================

    def predict(self,img):

        try:


            result = self.detector.findHands(
                img,
                draw=False
            )


            if isinstance(result,tuple):

                hands=result[0]
                img=result[1]

            else:

                hands=result



            if hands is None or len(hands)==0:

                return None,0,None,None



            all_x=[]
            all_y=[]



            for hand in hands:

                for lm in hand["lmList"]:

                    all_x.append(int(lm[0]))
                    all_y.append(int(lm[1]))



            pad=40


            h,w=img.shape[:2]


            x1=max(0,min(all_x)-pad)
            y1=max(0,min(all_y)-pad)

            x2=min(w,max(all_x)+pad)
            y2=min(h,max(all_y)+pad)



            crop = img[
                y1:y2,
                x1:x2
            ]


            if crop.size==0:

                return None,0,None,None



            inp,processed = self.preprocess_image(
                crop
            )


            prediction = self.model.predict(
                inp,
                verbose=0
            )



            index=np.argmax(
                prediction[0]
            )


            confidence=float(
                prediction[0][index]
            )


            label=self.idx_to_class[index]



            # DEBUG

            top3=np.argsort(
                prediction[0]
            )[-3:][::-1]


            print("\nTop 3 Predictions:")

            for i in top3:

                print(
                    f"{self.idx_to_class[i]} : {prediction[0][i]:.4f}"
                )



            # smoothing

            self.prediction_buffer.append(label)


            if len(self.prediction_buffer)>8:

                self.prediction_buffer.pop(0)



            stable_label = Counter(
                self.prediction_buffer
            ).most_common(1)[0][0]



            return (
                stable_label,
                confidence,
                (x1,y1,x2,y2),
                processed
            )



        except Exception as e:

            print("Prediction error:",e)

            return None,0,None,None





# ==========================================
# MAIN
# ==========================================

def main():


    cap=cv2.VideoCapture(0)


    cap.set(
        cv2.CAP_PROP_FRAME_WIDTH,
        1280
    )

    cap.set(
        cv2.CAP_PROP_FRAME_HEIGHT,
        720
    )


    detector=HandSignDetector()



    while True:


        success,img=cap.read()


        if not success:
            continue



        img=cv2.flip(
            img,
            1
        )



        label,confidence,bbox,processed = detector.predict(
            img
        )



        if bbox:


            x1,y1,x2,y2=bbox


            cv2.rectangle(
                img,
                (x1,y1),
                (x2,y2),
                (0,255,0),
                2
            )



        if label:


            # overlay right top

            cv2.putText(
                img,
                f"Alphabet: {label}",
                (img.shape[1]-350,70),
                cv2.FONT_HERSHEY_SIMPLEX,
                1.5,
                (0,255,0),
                3
            )


            cv2.putText(
                img,
                f"Confidence: {confidence*100:.1f}%",
                (img.shape[1]-350,130),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0,255,255),
                2
            )


            # voice

            detector.speak(label)




        cv2.imshow(
            "Sign Alphabet Recognition",
            img
        )



        if processed is not None:

            cv2.imshow(
                "Model Input",
                processed
            )



        if cv2.waitKey(1)==ord('q'):

            break



    cap.release()
    cv2.destroyAllWindows()



if __name__=="__main__":

    main()

Loading MobileNetV2 model...
Model loaded successfully!

Class Mapping:
{0: 'a', 1: 'b', 2: 'c', 3: 'd', 4: 'e', 5: 'f', 6: 'g', 7: 'h', 8: 'i', 9: 'j', 10: 'k', 11: 'l', 12: 'm', 13: 'n', 14: 'o', 15: 'p', 16: 'q', 17: 'r', 18: 's', 19: 't', 20: 'u', 21: 'v', 22: 'w', 23: 'x', 24: 'y', 25: 'z'}

Top 3 Predictions:
b : 0.8391
e : 0.1535
q : 0.0050

Top 3 Predictions:
w : 0.8927
b : 0.1060
x : 0.0007

Top 3 Predictions:
w : 0.9244
x : 0.0413
b : 0.0290

Top 3 Predictions:
w : 0.9866
b : 0.0083
a : 0.0043

Top 3 Predictions:
b : 0.5806
w : 0.2864
x : 0.0889

Top 3 Predictions:
w : 0.4927
b : 0.4784
a : 0.0232

Top 3 Predictions:
w : 0.6761
a : 0.1638
b : 0.1565

Top 3 Predictions:
b : 0.6955
w : 0.2894
p : 0.0093

Top 3 Predictions:
w : 0.9474
b : 0.0434
a : 0.0089

Top 3 Predictions:
w : 0.6250
b : 0.2303
a : 0.1445

Top 3 Predictions:
w : 0.7232
a : 0.1825
x : 0.0664

Top 3 Predictions:
w : 0.9910
b : 0.0090
x : 0.0000

Top 3 Predictions:
w : 0.6822
b : 0.2618
a : 0.0520

Top 3 Predict